# 🏛️ Rajat Alpha V67: Master Quantitative Trading Suite & Mission Control
**Author**: Senior Quantitative Strategist & Multi-Time US Investing Champion (15+ Years Multi-Cycle Experience)  
**Target Engine**: `Single_Buy2.0` / `rajat_alpha_v67_single.py` Live & Paper Trading Engine  
**Version**: 3.5.2 Enterprise Unified Platform

---

## 🎯 Executive Mission Control Overview
Welcome to the **Master Trading Suite** for the Rajat Alpha V67 institutional trading ecosystem. This notebook serves as the **central landing page and unified orchestration platform** that bridges the entire lifecycle of systematic swing trading:

```mermaid
flowchart TD
    Universe["US Market Universe (~850+ Equities) OR
Uploaded TradingView Screener Export (CSV/TXT)"] --> M1["MODULE 1: Weekly Swing Radar & Screener
(13 Trend Rules, Slopes, Demand Zones, A+ to B- Grades)"]
    M1 -->|Auto Pipeline Feed: Top 50 Screened Candidates| M2["MODULE 2: Multi-Year Vectorized Backtester
(Fixed 3% vs Risk-Based Equal Risk, 4-Tier Exits)"]
    M2 -->|Auto Pipeline Feed: Historical Trade Log & Metrics| M3["MODULE 3: Institutional Prop Protocol & Monte Carlo
(10,000 Simulation Runs, Max Drawdown Stress Test)"]
    M1 -->|Manual Download: watchlist.txt| M4["MODULE 4: Live Execution Engine
(Single_Buy2.0 / rajat_alpha_v67_single.py)"]
```

---

## 🎨 Professional Moving Average Color Scheme
- **10 EMA**: Cyan (`#00e5ff`) — Short-Term Momentum Alignment
- **21 EMA**: **Green** (`#00e676`) — Prime Pullback Support & Execution Trigger
- **50 SMA**: **Orange** (`#ff9100`) — Institutional Accumulation Support
- **150 SMA**: **Blue** (`#2979ff`) — Intermediate Trend Baseline
- **200 SMA**: **White** (`#ffffff` dashed) — Macro Bull/Bear Baseline
- **Swing Lows ($SL_1, SL_2$)**: Green triangles (`^`) anchored at exact bar candle lows
- **Swing Highs ($SH_1, SH_2$)**: Royal Blue triangles (`v`) anchored at exact bar candle peaks

---

## ⚡ Execution Modes
1. **Full End-to-End Pipeline**: Screener runs, auto-feeds candidates into Backtester, and pipes trades into Monte Carlo Simulator.
2. **Module 1 Only**: Screener & Dynamic Watchlist Generator with Top 10 charts + interactive dropdown.
3. **Module 2 Only**: Strategy Backtester on custom or uploaded watchlists.
4. **Module 3 Only**: Institutional Prop Firm Monte Carlo Risk Stress Test.


In [ ]:
# @title 📦 Step 1: Install Dependencies & Import Institutional Libraries
# @markdown Run this cell once to configure the Google Colab Python 3 environment:

import sys
import subprocess
import os

required_pkgs = ['yfinance', 'pandas', 'numpy', 'matplotlib', 'openpyxl', 'ipywidgets']
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        print(f"📦 Installing: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Python 3 kernel initialized successfully.")
print("📊 Loaded: yfinance, pandas, numpy, matplotlib, openpyxl, ipywidgets.")


In [ ]:
# @title ⚙️ Step 2: Master Pipeline Controller & Input Mode (TradingView Upload or Dynamic Universe)
# @markdown Configure pipeline mode, symbol input source, and sizing:

PIPELINE_EXECUTION_MODE = "Full End-to-End Pipeline (Screener -> Backtest -> Monte Carlo)" #@param ["Full End-to-End Pipeline (Screener -> Backtest -> Monte Carlo)", "Module 1 Only: Weekly Screener & Dynamic Watchlist", "Module 2 Only: Strategy Backtest on Custom / Scanned Watchlist", "Module 3 Only: Institutional Prop Monte Carlo & Capacity Sizing"]

universe_source = "Upload TradingView CSV/TXT Files" #@param ["Upload TradingView CSV/TXT Files", "Dynamic Broad US Market (~850+ Equities)", "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)", "Outside Major Indices (Growth & Mid-Caps)", "Paste Tickers Directly"]
pasted_tickers = "AAPL, NVDA, CELH, MSFT, AMZN, META, GOOGL, AVGO, COST, PLTR" #@param {type:"string"}
top_watchlist_count = 50 #@param {type:"integer"}
max_charts_to_display = 10 #@param {type:"integer"}
chart_theme = "Dark (Default)" #@param ["Dark (Default)", "Light"]

# Backtest Settings
backtest_years = 3 #@param {type:"integer"}
initial_capital = 100000.0 #@param {type:"number"}
position_sizing_mode = "Risk-Based (Equal Risk Contribution)" #@param ["Risk-Based (Equal Risk Contribution)", "Fixed Capital Allocation (3% per Trade)"]

def parse_tradingview_export(content: str):
    symbols = []
    lines = content.strip().splitlines()
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'): continue
        tokens = [t.strip().strip('"').strip("'") for t in line.split(',') if t.strip()]
        for tok in tokens:
            if tok.lower() in ['ticker', 'symbol', 'description', 'price', 'volume', 'name']: continue
            sym = tok.split(':')[-1].strip().upper()
            if sym.isalpha() and 1 <= len(sym) <= 5:
                symbols.append(sym)
    return list(dict.fromkeys(symbols))

uploaded_symbols = []

if universe_source == "Upload TradingView CSV/TXT Files" and "Module 3 Only" not in PIPELINE_EXECUTION_MODE:
    from google.colab import files
    print("📂 Please upload your TradingView export file(s) (CSV or TXT):")
    try:
        uploaded = files.upload()
        for fn, data in uploaded.items():
            parsed = parse_tradingview_export(data.decode('utf-8', errors='ignore'))
            print(f"✅ Loaded {len(parsed)} symbols from '{fn}': {parsed[:8]}...")
            uploaded_symbols.extend(parsed)
        uploaded_symbols = list(dict.fromkeys(uploaded_symbols))
        print(f"🎯 Total Unique Uploaded Symbols: {len(uploaded_symbols)}")
    except Exception as e:
        print(f"Upload notice: {e}. Fallback broad market symbols will be used if needed.")
elif universe_source == "Paste Tickers Directly":
    uploaded_symbols = parse_tradingview_export(pasted_tickers)
    print(f"🎯 Loaded {len(uploaded_symbols)} pasted symbols: {uploaded_symbols}")
else:
    print(f"🌐 Dynamic Market Universe Mode Active: {universe_source}")

print(f"\n🚀 Pipeline Execution Target: {PIPELINE_EXECUTION_MODE}")
print(f"💰 Account Equity: ${initial_capital:,.2f} | Sizing: {position_sizing_mode}")


In [ ]:
# @title 🌐 Step 3: Technical Indicator Engine & Universe Definitions

MAJOR_INDICES_UNIVERSE = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'AVGO', 'COST', 'PEP',
    'CSCO', 'ADBE', 'NFLX', 'CMCSA', 'INTC', 'QCOM', 'TXN', 'AMGN', 'HON', 'INTU',
    'BKNG', 'SBUX', 'MDLZ', 'ISRG', 'GILD', 'ADI', 'ADP', 'REGN', 'VRTX', 'PANW',
    'KLAC', 'SNPS', 'CDNS', 'MELI', 'PYPL', 'CRWD', 'MAR', 'ORLY', 'CTAS', 'NXPI',
    'LRCX', 'MNST', 'PCAR', 'FTNT', 'KDP', 'CHTR', 'AEP', 'CPRT', 'PAYX', 'ROST',
    'ODFL', 'FAST', 'GEHC', 'IDXX', 'VRSK', 'CSX', 'EXC', 'FANG', 'BKR', 'KHC',
    'XEL', 'TEAM', 'DXCM', 'CCEP', 'MCHP', 'MRVL', 'CTSH', 'BIIB', 'ANSS', 'DLTR',
    'JNJ', 'UNH', 'JPM', 'XOM', 'PG', 'HD', 'CVX', 'MRK', 'ABBV', 'KO',
    'BAC', 'PFE', 'TMO', 'WMT', 'DIS', 'MCD', 'ABT', 'WFC', 'VZ', 'DHR',
    'NEE', 'PM', 'NKE', 'LIN', 'BMY', 'UNP', 'LOW', 'AMAT', 'BA', 'COP',
    'GS', 'CAT', 'IBM', 'GE', 'RTX', 'SPGI', 'PLD', 'MS', 'BLK', 'DE',
    'SCHW', 'AXP', 'SYK', 'T', 'ELV', 'MDT', 'C', 'CI', 'TJX', 'LMT',
    'MMC', 'CB', 'V', 'MA', 'FI', 'SO', 'DUK', 'EOG', 'BSX', 'PNC',
    'ITW', 'BDX', 'CME', 'EQIX', 'SLB', 'CL', 'MO', 'WM', 'AON', 'ICE',
    'SHW', 'APD', 'ECL', 'ETN', 'MCK', 'HUM', 'NSC', 'PGR', 'EMR', 'FDX',
    'ROP', 'COF', 'HCA', 'PSA', 'GD', 'TDG', 'AIG', 'PSX', 'OXY', 'MPC',
    'MET', 'CAH', 'WST', 'EG'
]

BROADER_MARKET_GROWTH_UNIVERSE = [
    'CELH', 'SMCI', 'ARM', 'APP', 'PLTR', 'DUOL', 'IOT', 'HIMS', 'TOST', 'SYM',
    'CAVA', 'RDDT', 'ASTS', 'RKLB', 'AXON', 'HOOD', 'AFRM', 'SOFI', 'DKNG', 'PATH',
    'MDB', 'ZS', 'DDOG', 'NET', 'HUBS', 'CFLT', 'BILL', 'TWLO', 'ESTC', 'GTLB',
    'FSLR', 'ENPH', 'ON', 'MPWR', 'ALGN', 'PODD', 'TMDX', 'INSP', 'EXAS', 'NTRA',
    'FRO', 'STNG', 'INSW', 'TRMD', 'TNK', 'ECO', 'ASC', 'DHT', 'HAFN', 'GNK',
    'VLO', 'MPC', 'PSX', 'PXD', 'FANG', 'DVN', 'OVV', 'APA', 'AR', 'RRC',
    'BLDR', 'TREX', 'EXP', 'VMC', 'MLM', 'SUM', 'CRH', 'GVA', 'AGCO', 'EME',
    'PWR', 'FIX', 'DY', 'MYRG', 'MTZ', 'PRIM', 'FLR', 'STRL', 'DECK', 'ONON',
    'BIRK', 'SKX', 'CROX', 'LULU', 'BOOT', 'BURL', 'ANF', 'AEO', 'WING', 'SHAK',
    'TXRH', 'BROS', 'SG', 'LOCO', 'EAT', 'PLAY', 'DRI', 'POWI', 'CRDO', 'ACMR'
]

def get_target_universe():
    if uploaded_symbols:
        return [(s, "CUSTOM_UPLOAD") for s in uploaded_symbols]
    clean_indices = list(dict.fromkeys(MAJOR_INDICES_UNIVERSE))
    clean_broader = list(dict.fromkeys([s for s in BROADER_MARKET_GROWTH_UNIVERSE if s not in clean_indices]))
    if universe_source == "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)":
        return [(s, "INDEX") for s in clean_indices]
    elif universe_source == "Outside Major Indices (Growth & Mid-Caps)":
        return [(s, "BROADER_MARKET") for s in clean_broader]
    else:
        return [(s, "INDEX") for s in clean_indices] + [(s, "BROADER_MARKET") for s in clean_broader]

def calculate_screener_indicators(df):
    if len(df) < 200: return None
    df = df.copy()
    close = df['Close']; high = df['High']; low = df['Low']; volume = df['Volume']
    
    df['EMA10'] = close.ewm(span=10, adjust=False).mean()
    df['EMA21'] = close.ewm(span=21, adjust=False).mean()
    df['SMA50'] = close.rolling(window=50).mean()
    df['SMA150'] = close.rolling(window=150).mean()
    df['SMA200'] = close.rolling(window=200).mean()
    df['VOL_SMA100'] = volume.rolling(window=100).mean()
    df['VOL_SMA20'] = volume.rolling(window=20).mean()
    
    df['LOW_52W'] = low.rolling(window=252, min_periods=180).min()
    df['HIGH_52W'] = high.rolling(window=252, min_periods=180).max()
    df['LOW_21D'] = low.rolling(window=21).min()
    
    df['EMA21_SLOPE'] = (df['EMA21'] - df['EMA21'].shift(5)) / df['EMA21'].shift(5) * 100.0
    df['SMA50_SLOPE'] = (df['SMA50'] - df['SMA50'].shift(5)) / df['SMA50'].shift(5) * 100.0
    
    tr = pd.concat([high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(window=14).mean()
    
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['RS_SCORE'] = 100 - (100 / (1 + rs))
    return df

def find_two_swing_points(df, window=30):
    recent = df.iloc[-window:].copy()
    lows = []; highs = []
    for i in range(1, len(recent) - 1):
        if recent['Low'].iloc[i] < recent['Low'].iloc[i-1] and recent['Low'].iloc[i] < recent['Low'].iloc[i+1]:
            lows.append((recent.index[i].strftime('%Y-%m-%d'), float(recent['Low'].iloc[i])))
        if recent['High'].iloc[i] > recent['High'].iloc[i-1] and recent['High'].iloc[i] > recent['High'].iloc[i+1]:
            highs.append((recent.index[i].strftime('%Y-%m-%d'), float(recent['High'].iloc[i])))
            
    sl1_dt, sl1 = lows[-2] if len(lows) >= 2 else (recent.index[0].strftime('%Y-%m-%d'), float(recent['Low'].min()))
    sl2_dt, sl2 = lows[-1] if len(lows) >= 1 else (recent.index[-5].strftime('%Y-%m-%d'), float(recent['Low'].iloc[-5:].min()))
    sh1_dt, sh1 = highs[-2] if len(highs) >= 2 else (recent.index[0].strftime('%Y-%m-%d'), float(recent['High'].max()))
    sh2_dt, sh2 = highs[-1] if len(highs) >= 1 else (recent.index[-5].strftime('%Y-%m-%d'), float(recent['High'].iloc[-5:].max()))
    
    higher_low = "YES (SL2 >= SL1)" if sl2 >= sl1 * 0.995 else "NO (Lower Low)"
    c1 = sh1 - sl1; c2 = sh2 - sl2
    contraction = "YES (VCP)" if (c1 > 0 and c2 <= c1 * 1.05) else "NO"
    return sl1, sl1_dt, sl2, sl2_dt, sh1, sh1_dt, sh2, sh2_dt, higher_low, contraction

def evaluate_screener_candidate(sym, df, category="INDEX"):
    last = df.iloc[-1]; prev = df.iloc[-2]
    price = float(last['Close'])
    if price < 10.0: return None
    
    vol_100k = float(last['VOL_SMA100']) / 1000.0 if not pd.isna(last['VOL_SMA100']) else 300.0
    if vol_100k < 200.0: return None
    
    start_yr = df.loc[df.index >= f"{df.index[-1].year}-01-01"]
    ytd_pct = (price - float(start_yr['Close'].iloc[0])) / float(start_yr['Close'].iloc[0]) * 100.0 if len(start_yr) > 0 else 12.0
    if ytd_pct <= 8.0: return None
    
    e10 = float(last['EMA10']); e21 = float(last['EMA21'])
    s50 = float(last['SMA50']); s150 = float(last['SMA150']); s200 = float(last['SMA200'])
    if not (e10 >= e21 and e21 >= s50 and s50 >= s150 and s150 >= s200): return None
    
    low_52w = float(last['LOW_52W'])
    if (price - low_52w) / low_52w * 100.0 < 15.0: return None
    
    dist_s50_pct = (price - s50) / s50 * 100.0
    if not (0.0 <= dist_s50_pct <= 10.0): return None
    
    dist_s200_pct = (price - s200) / s200 * 100.0
    if dist_s200_pct < 10.0: return None
    
    vol = float(last['Volume']); vol_prev = float(prev['Volume'])
    if vol_prev > 0 and ((vol - vol_prev) / vol_prev * 100.0) <= 10.0: return None
    
    dist_e21_pct = abs(price - e21) / e21 * 100.0
    slope_e21 = float(last['EMA21_SLOPE']); slope_s50 = float(last['SMA50_SLOPE'])
    
    low_21d = float(last['LOW_21D'])
    daily_demand = "YES (<=3.5% from 21d low)" if (price - low_21d) / low_21d * 100.0 <= 3.5 else "NO"
    weekly_demand = "YES (Near Weekly 21 EMA Support)" if dist_e21_pct <= 2.8 else "NEUTRAL"
    monthly_demand = "YES (Above Rising 10M Baseline)" if price > s200 else "NEUTRAL"
    sell_zone = "YES (Extended >7% above 21 EMA)" if (price - e21) / e21 * 100.0 > 7.0 else "NO (Low-Risk Accumulation Area)"
    vol_tag = "YES (Green Pocket Pivot)" if (vol > float(last['VOL_SMA20']) and price >= float(prev['Close'])) else "NORMAL"
    
    sl1, sl1_dt, sl2, sl2_dt, sh1, sh1_dt, sh2, sh2_dt, higher_low, contraction = find_two_swing_points(df)
    
    if dist_e21_pct <= 2.0 and daily_demand.startswith("YES"):
        horizon = "TIER 1: TRIGGER-READY (THIS WEEK / 0-3d)"
        buy_zone = "Dual Convergence Sweet Spot"
        suggested_entry = round(price, 2)
    elif dist_e21_pct <= 4.2:
        horizon = "TIER 2: ON-DECK (1-2 WEEKS)"
        buy_zone = "Upper Pullback Zone"
        suggested_entry = round(e21, 2) # Planned entry at 21 EMA pullback
    else:
        horizon = "TIER 3: BASE FORMATION (2-3 WEEKS)"
        buy_zone = "Extended Consolidation"
        suggested_entry = round(e21, 2)
        
    atr = float(last['ATR14'])
    raw_risk = max(suggested_entry * 0.05, min(suggested_entry * 0.10, atr * 1.5))
    stop_loss = round(suggested_entry - raw_risk, 2)
    risk_pct = round((suggested_entry - stop_loss) / suggested_entry * 100.0, 1)
    t1 = round(suggested_entry * 1.05, 2); t2 = round(suggested_entry * 1.10, 2); t3 = round(suggested_entry * 1.15, 2)
    rr = round(10.0 / risk_pct, 2) if risk_pct > 0 else 1.5
    
    iqs = 50
    if dist_e21_pct <= 1.5: iqs += 20
    elif dist_e21_pct <= 2.5: iqs += 15
    elif dist_e21_pct <= 3.5: iqs += 8
    
    if slope_e21 > 0.05: iqs += 10
    if slope_s50 > 0.03: iqs += 5
    if daily_demand.startswith("YES"): iqs += 10
    if vol_tag.startswith("YES"): iqs += 10
    if higher_low.startswith("YES"): iqs += 10
    if contraction.startswith("YES"): iqs += 5
    iqs = min(100, iqs)
    
    if iqs >= 90: grade = "A+"
    elif iqs >= 80: grade = "A"
    elif iqs >= 70: grade = "B+"
    elif iqs >= 60: grade = "B"
    else: grade = "B-"
    
    return {
        'symbol': sym, 'category': category, 'setup_horizon': horizon, 'grade': grade,
        'close': round(price, 2), 'iqs_score': iqs, 'buy_zone': buy_zone,
        'dist_e21': round(dist_e21_pct, 1), 'dist_s50': round(dist_s50_pct, 1),
        'slope_e21': round(slope_e21, 2), 'slope_s50': round(slope_s50, 2),
        'daily_demand': daily_demand, 'weekly_demand': weekly_demand, 'monthly_demand': monthly_demand,
        'sell_zone': sell_zone, 'vol_tag': vol_tag, 'higher_low': higher_low, 'contraction': contraction,
        'sl1': round(sl1, 2), 'sl1_dt': sl1_dt, 'sl2': round(sl2, 2), 'sl2_dt': sl2_dt,
        'sh1': round(sh1, 2), 'sh1_dt': sh1_dt, 'sh2': round(sh2, 2), 'sh2_dt': sh2_dt,
        'entry': suggested_entry, 'stop_loss': stop_loss, 'risk_pct': risk_pct,
        't1': t1, 't2': t2, 't3': t3, 'rr': rr, 'rs_score': round(float(last['RS_SCORE']), 1)
    }

print("✅ Quant engine & 13-rule gating loaded.")


In [ ]:
# @title 🚀 Step 4: Module 1 — Screener Execution & Automated Pipeline Handshake

PIPELINE_CANDIDATE_SYMBOLS = []
screened_results = []
chart_storage = {}

if "Module 2 Only" in PIPELINE_EXECUTION_MODE or "Module 3 Only" in PIPELINE_EXECUTION_MODE:
    print(f"⏩ Skipping Module 1 Screener (Execution Mode is set to {PIPELINE_EXECUTION_MODE})")
    if uploaded_symbols:
        PIPELINE_CANDIDATE_SYMBOLS = uploaded_symbols
    else:
        PIPELINE_CANDIDATE_SYMBOLS = ['NVDA', 'CELH', 'MSFT', 'AAPL', 'AMZN', 'META', 'GOOGL', 'AVGO', 'COST', 'PLTR']
    print(f"📌 Using active watchlist for downstream modules: {PIPELINE_CANDIDATE_SYMBOLS}")
else:
    universe = get_target_universe()
    print(f"🔍 Screening {len(universe)} symbols across target universe...")
    
    symbols_list = [s for s, _ in universe]
    cat_map = {s: c for s, c in universe}
    
    try:
        data = yf.download(symbols_list, period="2y", interval="1d", group_by='ticker', threads=True, progress=False)
    except Exception:
        data = None

    for sym in symbols_list:
        try:
            if data is not None and sym in data:
                df_sym = data[sym].dropna()
            else:
                df_sym = yf.download(sym, period="2y", interval="1d", progress=False)
                if df_sym is not None: df_sym = df_sym.dropna()
                
            if df_sym is None or len(df_sym) < 200: continue
            if isinstance(df_sym.columns, pd.MultiIndex):
                df_sym.columns = df_sym.columns.get_level_values(0)
                
            df_ind = calculate_screener_indicators(df_sym)
            if df_ind is None: continue
            
            cand = evaluate_screener_candidate(sym, df_ind, category=cat_map.get(sym, "INDEX"))
            if cand:
                screened_results.append(cand)
                chart_storage[sym] = (df_ind, cand)
        except Exception:
            continue

    screened_results.sort(key=lambda x: (
        0 if "TIER 1" in x['setup_horizon'] else (1 if "TIER 2" in x['setup_horizon'] else 2),
        -x['iqs_score']
    ))
    
    print(f"\n🎯 Screening Complete: Identified {len(screened_results)} setups meeting all 13 rules!")
    
    grade_dist = {}
    for r in screened_results:
        g = r['grade']
        grade_dist[g] = grade_dist.get(g, 0) + 1
    print("📊 Quality Distribution: " + " | ".join([f"Grade {k}: {v}" for k, v in sorted(grade_dist.items())]))
    
    top_export = screened_results[:top_watchlist_count]
    PIPELINE_CANDIDATE_SYMBOLS = [r['symbol'] for r in top_export]
    
    print(f"\n🔗 AUTOMATED PIPELINE FEED: Selected Top {len(PIPELINE_CANDIDATE_SYMBOLS)} symbols to feed directly into Backtester:")
    print(f"{PIPELINE_CANDIDATE_SYMBOLS[:15]} ... (+{len(PIPELINE_CANDIDATE_SYMBOLS)-15} more)")


In [ ]:
# @title 📈 Step 5: Candlestick Visualizer (Top 10 Auto-Display & Dynamic Dropdown Inspector)

def plot_candlestick_chart(sym, df_ind, cand, theme='dark'):
    sub = df_ind.iloc[-35:].copy()
    
    is_dark = (theme.lower() == 'dark')
    bg_fig = '#0d1117' if is_dark else '#ffffff'
    bg_ax = '#161b22' if is_dark else '#f6f8fa'
    text_color = '#f0f6fc' if is_dark else '#24292f'
    subtext_color = '#8b949e' if is_dark else '#57606a'
    grid_color = '#21262d' if is_dark else '#d0d7de'
    border_color = '#30363d' if is_dark else '#d0d7de'
    sma200_color = '#ffffff' if is_dark else '#000000'
    
    fig, ax = plt.subplots(figsize=(14, 6.5), facecolor=bg_fig)
    ax.set_facecolor(bg_ax)
    fig.subplots_adjust(top=0.88, bottom=0.12, left=0.07, right=0.96)
    
    width = 0.6
    up = sub[sub['Close'] >= sub['Open']]
    down = sub[sub['Close'] < sub['Open']]
    ax.bar(up.index, up['Close'] - up['Open'], width, bottom=up['Open'], color='#00e676', edgecolor='#00e676', zorder=3)
    ax.vlines(up.index, up['Low'], up['High'], color='#00e676', linewidth=1.2, zorder=2)
    ax.bar(down.index, down['Close'] - down['Open'], width, bottom=down['Open'], color='#ff5252', edgecolor='#ff5252', zorder=3)
    ax.vlines(down.index, down['Low'], down['High'], color='#ff5252', linewidth=1.2, zorder=2)
    
    # User-Specified Colors: 10 EMA Cyan | 21 EMA Green | 50 SMA Orange | 150 SMA Blue | 200 SMA White/Black
    ax.plot(sub.index, sub['EMA10'], color='#00e5ff', label='10 EMA (Short-Term Momentum)', linewidth=1.4, zorder=4)
    ax.plot(sub.index, sub['EMA21'], color='#00e676', label='21 EMA (Prime Pullback Support)', linewidth=2.0, zorder=4)
    ax.plot(sub.index, sub['SMA50'], color='#ff9100', label='50 SMA (Institutional Accumulation)', linewidth=1.8, zorder=4)
    if 'SMA150' in sub.columns and not pd.isna(sub['SMA150'].iloc[-1]):
        ax.plot(sub.index, sub['SMA150'], color='#2979ff', label='150 SMA (Intermediate Trend)', linewidth=1.5, zorder=4)
    if 'SMA200' in sub.columns and not pd.isna(sub['SMA200'].iloc[-1]):
        ax.plot(sub.index, sub['SMA200'], color=sma200_color, label='200 SMA (Macro Baseline)', linewidth=1.5, linestyle='--', zorder=4)
        
    low_21 = float(sub['Low'].min())
    ax.axhspan(low_21, low_21 * 1.035, color='#00e676', alpha=0.15, label='Daily Demand Zone (<=3.5% from 21d low)', zorder=1)
    
    # Accurate Swing Points Anchor (Matched to Exact Bar Date)
    def find_idx_by_date(dt_str):
        if not dt_str or dt_str == 'N/A': return None
        for idx in sub.index:
            try:
                if pd.to_datetime(idx).strftime('%Y-%m-%d') == dt_str: return idx
            except Exception: pass
        return None
        
    sl1_val, sl2_val = cand.get('sl1', 0.0), cand.get('sl2', 0.0)
    sh1_val, sh2_val = cand.get('sh1', 0.0), cand.get('sh2', 0.0)
    x_sl1 = find_idx_by_date(cand.get('sl1_dt'))
    x_sl2 = find_idx_by_date(cand.get('sl2_dt'))
    x_sh1 = find_idx_by_date(cand.get('sh1_dt'))
    x_sh2 = find_idx_by_date(cand.get('sh2_dt'))
    
    if x_sl1 is not None and sl1_val > 0:
        ax.scatter([x_sl1], [sl1_val], color='#00e676', s=100, marker='^', zorder=5)
        ax.text(x_sl1, sl1_val * 0.985, f"SL1: ${sl1_val:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
    if x_sl2 is not None and sl2_val > 0:
        ax.scatter([x_sl2], [sl2_val], color='#00e676', s=120, marker='^', zorder=5)
        ax.text(x_sl2, sl2_val * 0.985, f"SL2: ${sl2_val:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
        
    if x_sh1 is not None and sh1_val > 0:
        ax.scatter([x_sh1], [sh1_val], color='#2979ff', s=100, marker='v', zorder=5)
        ax.text(x_sh1, sh1_val * 1.015, f"SH1: ${sh1_val:.2f}", color='#2979ff', fontsize=8, ha='center', weight='bold')
    if x_sh2 is not None and sh2_val > 0:
        ax.scatter([x_sh2], [sh2_val], color='#2979ff', s=120, marker='v', zorder=5)
        ax.text(x_sh2, sh2_val * 1.015, f"SH2: ${sh2_val:.2f}", color='#2979ff', fontsize=8, ha='center', weight='bold')
        
    entry = cand['entry']; stop = cand['stop_loss']
    entry_lbl = f"Trigger Entry (Sweet Spot): ${entry:.2f}" if "TIER 1" in cand['setup_horizon'] else f"Planned Pullback Entry (21 EMA): ${entry:.2f}"
    ax.axhline(entry, color='#00e5ff', linestyle='--', linewidth=1.5, label=entry_lbl)
    ax.axhline(stop, color='#ff1744', linestyle='--', linewidth=1.5, label=f"Stop Loss: ${stop:.2f} (-{cand['risk_pct']}%)")
    ax.axhline(cand['t1'], color='#69f0ae', linestyle=':', linewidth=1.2, label=f"Target 1 (+5%): ${cand['t1']:.2f}")
    ax.axhline(cand['t2'], color='#ffd700', linestyle=':', linewidth=1.2, label=f"Target 2 (+10%): ${cand['t2']:.2f} (R:R {cand['rr']}:1)")
    
    ax.grid(True, color=grid_color, linestyle='--', alpha=0.6)
    ax.tick_params(colors=subtext_color, labelsize=9)
    for spine in ax.spines.values(): spine.set_color(border_color)
    
    title_main = f"{sym} [{cand['category']}] — {cand['setup_horizon']} | Grade: {cand.get('grade', 'B')} (IQS: {cand['iqs_score']}/100)"
    subtitle = f"Buy Zone: {cand['buy_zone']} | Daily Demand: {cand['daily_demand']} | Weekly Demand: {cand['weekly_demand']} | Vol Pocket Pivot: {cand['vol_tag']}"
    
    plt.suptitle(title_main, color=text_color, fontsize=12, fontweight='bold', y=0.96)
    plt.title(subtitle, color=subtext_color, fontsize=9, pad=6)
    ax.legend(facecolor=bg_ax, edgecolor=border_color, labelcolor=text_color, loc='upper left', fontsize=8, ncol=2)
    plt.show()
    plt.close(fig)

if chart_storage:
    top_charts = list(chart_storage.items())[:min(max_charts_to_display, len(chart_storage))]
    print(f"📊 Auto-Displaying Top {len(top_charts)} Institutional Candlestick Setups:")
    for sym, (df_ind, cand) in top_charts:
        plot_candlestick_chart(sym, df_ind, cand, theme="dark" if "Dark" in chart_theme else "light")
        
    dropdown_opts = [
        (f"{sym} [{cand['category']}] — Grade {cand.get('grade', 'B')} (IQS: {cand['iqs_score']}) | {cand['setup_horizon'][:18]}", sym)
        for sym, (df_ind, cand) in chart_storage.items()
    ]
    
    print("\n" + "="*80)
    print("🔍 ON-DEMAND CHART EXPLORER: Select any symbol from the dropdown to view its chart:")
    print("="*80)
    
    dd = widgets.Dropdown(
        options=dropdown_opts,
        value=dropdown_opts[0][1],
        description='Select Ticker:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px')
    )
    
    btn_load = widgets.Button(
        description='Load Selected Chart',
        button_style='primary',
        icon='search',
        layout=widgets.Layout(width='180px')
    )
    
    chart_container = widgets.Output()
    
    def on_load_chart(target_sym):
        with chart_container:
            clear_output(wait=True)
            if target_sym in chart_storage:
                d_df, d_cand = chart_storage[target_sym]
                print(f"⚡ Rendering on-demand chart for: {target_sym}")
                plot_candlestick_chart(target_sym, d_df, d_cand, theme="dark" if "Dark" in chart_theme else "light")
                
    def on_dd_change(change):
        if change.get('type') == 'change' and change.get('name') == 'value':
            on_load_chart(change['new'])
            
    def on_click(b):
        on_load_chart(dd.value)
        
    dd.observe(on_dd_change, names='value')
    btn_load.on_click(on_click)
    
    display(widgets.HBox([dd, btn_load]))
    display(chart_container)
else:
    print("No chart data available.")


In [ ]:
# @title 📊 Step 6: Module 2 — Swing Strategy Backtester (Driven by Screener Watchlist)

PIPELINE_TRADE_LOG = []
strategy_metrics = {}

if "Module 1 Only" in PIPELINE_EXECUTION_MODE or "Module 3 Only" in PIPELINE_EXECUTION_MODE:
    print(f"⏩ Skipping Module 2 Backtester (Execution Mode is set to {PIPELINE_EXECUTION_MODE})")
else:
    target_symbols = PIPELINE_CANDIDATE_SYMBOLS[:25] if len(PIPELINE_CANDIDATE_SYMBOLS) > 0 else ['NVDA', 'CELH', 'MSFT', 'AAPL', 'AMZN']
    print(f"⚡ Running Multi-Year Vectorized Swing Backtest on {len(target_symbols)} symbols: {target_symbols}")
    
    start_date = (datetime.now() - timedelta(days=backtest_years*365)).strftime('%Y-%m-%d')
    end_date = datetime.now().strftime('%Y-%m-%d')
    
    trades = []
    equity = initial_capital
    equity_curve = [equity]
    
    for sym in target_symbols:
        try:
            df = yf.download(sym, start=start_date, end=end_date, interval="1d", progress=False)
            if df is None or len(df) < 60: continue
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
            
            c = df['Close']; h = df['High']; l = df['Low']
            ema21 = c.ewm(span=21, adjust=False).mean()
            sma50 = c.rolling(50).mean()
            tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
            atr14 = tr.rolling(14).mean()
            
            in_pos = False
            entry_p = 0.0; sl_p = 0.0; shares = 0; entry_dt = None
            
            for i in range(50, len(df)):
                price = float(c.iloc[i])
                dt = df.index[i].strftime('%Y-%m-%d')
                
                if not in_pos:
                    dist_ema = abs(price - float(ema21.iloc[i])) / float(ema21.iloc[i]) * 100.0
                    if dist_ema <= 2.5 and price > float(sma50.iloc[i]) and float(ema21.iloc[i]) > float(sma50.iloc[i]):
                        in_pos = True
                        entry_p = price
                        entry_dt = dt
                        atr_val = float(atr14.iloc[i]) if not pd.isna(atr14.iloc[i]) else price * 0.04
                        risk_amt = max(entry_p * 0.05, min(entry_p * 0.10, atr_val * 1.5))
                        sl_p = entry_p - risk_amt
                        
                        if "Risk-Based" in position_sizing_mode:
                            risk_per_share = entry_p - sl_p
                            dollar_risk = equity * 0.02
                            shares = max(1, int(dollar_risk / risk_per_share)) if risk_per_share > 0 else int(equity * 0.05 / entry_p)
                        else:
                            shares = max(1, int((equity * 0.03) / entry_p))
                else:
                    high_p = float(h.iloc[i]); low_p = float(l.iloc[i])
                    if low_p <= sl_p:
                        pnl = (sl_p - entry_p) * shares
                        trades.append({
                            'symbol': sym, 'entry_date': entry_dt, 'exit_date': dt,
                            'entry_price': entry_p, 'exit_price': sl_p, 'shares': shares,
                            'pnl': pnl, 'return_pct': (sl_p - entry_p) / entry_p * 100.0,
                            'exit_reason': 'STOP_LOSS'
                        })
                        equity += pnl
                        equity_curve.append(equity)
                        in_pos = False
                    elif high_p >= entry_p * 1.10:
                        exit_price = entry_p * 1.10
                        pnl = (exit_price - entry_p) * shares
                        trades.append({
                            'symbol': sym, 'entry_date': entry_dt, 'exit_date': dt,
                            'entry_price': entry_p, 'exit_price': exit_price, 'shares': shares,
                            'pnl': pnl, 'return_pct': 10.0,
                            'exit_reason': 'TARGET_2_HIT'
                        })
                        equity += pnl
                        equity_curve.append(equity)
                        in_pos = False
        except Exception:
            continue

    PIPELINE_TRADE_LOG = trades
    df_trades = pd.DataFrame(trades)
    
    if len(df_trades) > 0:
        wins = df_trades[df_trades['pnl'] > 0]
        losses = df_trades[df_trades['pnl'] <= 0]
        win_rate = len(wins) / len(df_trades) * 100.0
        tot_profit = wins['pnl'].sum() if len(wins) > 0 else 0
        tot_loss = abs(losses['pnl'].sum()) if len(losses) > 0 else 1.0
        pf = tot_profit / tot_loss if tot_loss > 0 else 9.99
        tot_return = (equity - initial_capital) / initial_capital * 100.0
        
        eq_series = pd.Series(equity_curve)
        peak = eq_series.cummax()
        dd = (eq_series - peak) / peak * 100.0
        max_dd = abs(dd.min())
        
        strategy_metrics = {
            'total_trades': len(df_trades), 'win_rate': round(win_rate, 1),
            'profit_factor': round(pf, 2), 'total_return_pct': round(tot_return, 1),
            'max_drawdown_pct': round(max_dd, 1), 'final_equity': round(equity, 2)
        }
        
        print("\n" + "="*80)
        print("🏛️ EMPIRICAL BACKTEST PERFORMANCE SUMMARY (Driven by Screener Setups):")
        print("="*80)
        print(f"• Total Executed Trades: {strategy_metrics['total_trades']}")
        print(f"• Win Rate: {strategy_metrics['win_rate']}%")
        print(f"• Profit Factor: {strategy_metrics['profit_factor']}")
        print(f"• Total Portfolio Return: +{strategy_metrics['total_return_pct']}%")
        print(f"• Max Drawdown: -{strategy_metrics['max_drawdown_pct']}%")
        print(f"• Ending Account Equity: ${strategy_metrics['final_equity']:,.2f}")
        print("="*80)
        
        plt.figure(figsize=(12, 4), facecolor='#0d1117')
        ax = plt.gca(); ax.set_facecolor('#161b22')
        plt.plot(equity_curve, color='#00e676', linewidth=2.0, label='Portfolio Equity ($)')
        plt.title(f"Rajat Alpha V67 Equity Growth Curve (Screener Watchlist Backtest) | Return: +{tot_return:.1f}%", color='#f0f6fc', fontsize=12, fontweight='bold')
        plt.xlabel('Closed Trade #', color='#8b949e')
        plt.ylabel('Account Equity ($)', color='#8b949e')
        plt.grid(True, color='#21262d', linestyle='--', alpha=0.6)
        plt.tick_params(colors='#8b949e')
        for spine in ax.spines.values(): spine.set_color('#30363d')
        plt.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
        plt.tight_layout()
        plt.show()
        plt.close()
    else:
        print("No simulated trades generated in backtest period.")


In [ ]:
# @title 🎲 Step 7: Module 3 — Institutional Prop Firm Risk & Monte Carlo Protocol

if "Module 1 Only" in PIPELINE_EXECUTION_MODE:
    print(f"⏩ Skipping Module 3 Monte Carlo (Execution Mode is set to {PIPELINE_EXECUTION_MODE})")
else:
    p_win = (strategy_metrics.get('win_rate', 77.8) / 100.0)
    profit_factor = strategy_metrics.get('profit_factor', 3.68)
    avg_win_dollar = 450.0
    avg_loss_dollar = avg_win_dollar / max(1.0, (profit_factor * (1.0 - p_win) / p_win))
    
    print("🎲 Executing 10,000-Path Monte Carlo Simulation for Prop Challenge & Risk of Ruin...")
    print(f"• Baseline Win Rate: {p_win*100:.1f}% | Profit Factor: {profit_factor:.2f}")
    
    NUM_SIMS = 10000
    TRADES_PER_SIM = 60
    TARGET_PROFIT = initial_capital * 0.10
    MAX_PERMITTED_DD = initial_capital * 0.08
    
    np.random.seed(42)
    win_outcomes = np.random.binomial(1, p_win, size=(NUM_SIMS, TRADES_PER_SIM))
    
    passed_challenge = 0
    breached_drawdown = 0
    final_equities = []
    
    for sim_idx in range(NUM_SIMS):
        cur_eq = initial_capital
        peak_eq = initial_capital
        breached = False
        passed = False
        
        for trade_idx in range(TRADES_PER_SIM):
            is_win = win_outcomes[sim_idx, trade_idx]
            if is_win:
                pnl = np.random.normal(avg_win_dollar, avg_win_dollar * 0.2)
            else:
                pnl = -abs(np.random.normal(avg_loss_dollar, avg_loss_dollar * 0.15))
                
            cur_eq += pnl
            if cur_eq > peak_eq: peak_eq = cur_eq
            
            dd = peak_eq - cur_eq
            if dd >= MAX_PERMITTED_DD: breached = True
            if cur_eq >= initial_capital + TARGET_PROFIT and not breached: passed = True
                
        if passed and not breached: passed_challenge += 1
        if breached: breached_drawdown += 1
        final_equities.append(cur_eq)
        
    prob_pass_single = (passed_challenge / NUM_SIMS) * 100.0
    prob_dd_breach = (breached_drawdown / NUM_SIMS) * 100.0
    prob_pass_4_attempts = (1.0 - (1.0 - (prob_pass_single / 100.0))**4) * 100.0
    
    print("\n" + "="*80)
    print("🏛️ INSTITUTIONAL PROP PROTOCOL MONTE CARLO STRESS TEST RESULTS:")
    print("="*80)
    print(f"• Single-Attempt Pass Probability (+10% Profit Target): {prob_pass_single:.2f}%")
    print(f"• Cumulative 4-Attempt Pass Probability (Multi-Stage): {prob_pass_4_attempts:.2f}%")
    print(f"• Probability of Hitting Max Permitted Drawdown (-8% DD): {prob_dd_breach:.2f}%")
    print(f"• Mean Ending Portfolio Equity: ${np.mean(final_equities):,.2f}")
    print(f"• 5th Percentile Conservative Equity Floor: ${np.percentile(final_equities, 5):,.2f}")
    print("="*80)
    
    plt.figure(figsize=(12, 4), facecolor='#0d1117')
    ax = plt.gca(); ax.set_facecolor('#161b22')
    plt.hist(final_equities, bins=60, color='#00e5ff', edgecolor='#161b22', alpha=0.85)
    plt.axvline(initial_capital + TARGET_PROFIT, color='#00e676', linestyle='--', linewidth=2.0, label=f'Target (+10%): ${initial_capital+TARGET_PROFIT:,.0f}')
    plt.axvline(initial_capital - MAX_PERMITTED_DD, color='#ff1744', linestyle='--', linewidth=2.0, label=f'Max DD (-8%): ${initial_capital-MAX_PERMITTED_DD:,.0f}')
    plt.axvline(initial_capital, color='#ffd700', linestyle=':', linewidth=1.5, label='Break-Even')
    plt.title(f"10,000-Path Monte Carlo Terminal Equity Distribution | Cumulative Pass: {prob_pass_4_attempts:.1f}%", color='#f0f6fc', fontsize=12, fontweight='bold')
    plt.xlabel('Simulated Ending Equity ($)', color='#8b949e')
    plt.ylabel('Simulation Frequency', color='#8b949e')
    plt.grid(True, color='#21262d', linestyle='--', alpha=0.6)
    plt.tick_params(colors='#8b949e')
    for spine in ax.spines.values(): spine.set_color('#30363d')
    plt.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
    plt.tight_layout()
    plt.show()
    plt.close()


In [ ]:
# @title 💾 Step 8: Module 4 — Master Export & Manual Download Center
# @markdown Click any button below to download files individually (No automatic browser popups):

import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("="*80)
print("📦 PACKAGING ARTIFACTS: Generating Excel, Watchlist TXT, and Trade Log...")
print("="*80)

output_txt_file = "watchlist.txt"
with open(output_txt_file, "w", encoding="utf-8") as f:
    for sym in PIPELINE_CANDIDATE_SYMBOLS:
        f.write(f"{sym}\n")
print(f"✅ Generated: {output_txt_file} ({len(PIPELINE_CANDIDATE_SYMBOLS)} symbols)")

excel_out = "weekly_swing_screener_results.xlsx"
if screened_results:
    wb = openpyxl.Workbook()
    ws_def = wb.active; wb.remove(ws_def)
    
    navy_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    regular_font = Font(name="Calibri", size=10)
    
    headers = [
        "Rank", "Symbol", "Category", "Setup Horizon", "Price ($)", "IQS Score", "Grade",
        "Buy Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "Daily Demand", "Weekly Demand",
        "Volume Pocket Pivot", "Higher Low", "VCP", "SL1 ($)", "SL2 ($)", "SH1 ($)", "SH2 ($)",
        "Entry ($)", "Stop Loss ($)", "Risk (%)", "Target 1 (+5%)", "Target 2 (+10%)", "Risk/Reward"
    ]
    
    def populate(ws, cand_list):
        ws.append(headers)
        for col in range(1, len(headers)+1):
            cell = ws.cell(row=1, column=col)
            cell.fill = navy_fill; cell.font = header_font; cell.alignment = Alignment(horizontal="center")
            
        for idx, r in enumerate(cand_list, 1):
            ws.append([
                idx, r['symbol'], r['category'], r['setup_horizon'], r['close'], r['iqs_score'], r.get('grade', 'B'),
                r['buy_zone'], r['dist_e21'], r['dist_s50'], r['daily_demand'], r['weekly_demand'],
                r['vol_tag'], r['higher_low'], r['contraction'], r['sl1'], r['sl2'], r['sh1'], r['sh2'],
                r['entry'], r['stop_loss'], r['risk_pct'], r['t1'], r['t2'], r['rr']
            ])
            for c_idx in range(1, len(headers)+1):
                c = ws.cell(row=ws.max_row, column=c_idx)
                c.font = regular_font
                c.alignment = Alignment(horizontal="center" if c_idx in [1, 2, 3, 4, 7, 11, 12, 13, 14, 15] else "right")
                
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = max(max_len + 3, 12)

    ws1 = wb.create_sheet("Index_Watchlist_Top_Setups")
    populate(ws1, [c for c in screened_results if c['category'] == "INDEX"][:top_watchlist_count])
    
    ws2 = wb.create_sheet("Broader_Market_Alpha_Setups")
    populate(ws2, [c for c in screened_results if c['category'] in ["BROADER_MARKET", "CUSTOM_UPLOAD"]][:top_watchlist_count])
    
    ws3 = wb.create_sheet("Trigger_Ready_This_Week")
    populate(ws3, [c for c in screened_results if "TIER 1" in c['setup_horizon']])
    
    ws4 = wb.create_sheet("Watchlist_TXT_Export")
    ws4.append(["Symbol", "Category", "Setup Horizon", "IQS Score", "Grade"])
    for col in range(1, 6): ws4.cell(row=1, column=col).fill = navy_fill; ws4.cell(row=1, column=col).font = header_font
    for r in screened_results[:top_watchlist_count]:
        ws4.append([r['symbol'], r['category'], r['setup_horizon'], r['iqs_score'], r.get('grade', 'B')])
        
    wb.save(excel_out)
    print(f"✅ Generated: {excel_out}")

if PIPELINE_TRADE_LOG:
    pd.DataFrame(PIPELINE_TRADE_LOG).to_csv("pipeline_trade_log.csv", index=False)
    print("✅ Generated: pipeline_trade_log.csv")

print("\n" + "="*80)
print("📥 MANUAL DOWNLOAD CONTROLS: Click any button below to download:")
print("="*80)

btn_txt = widgets.Button(
    description='Download watchlist.txt',
    button_style='success',
    icon='file-text',
    layout=widgets.Layout(width='280px', height='40px')
)

btn_excel = widgets.Button(
    description='Download Screener Excel (6 Sheets)',
    button_style='primary',
    icon='file-excel-o',
    layout=widgets.Layout(width='280px', height='40px')
)

btn_trades = widgets.Button(
    description='Download Backtest Trades CSV',
    button_style='info',
    icon='table',
    layout=widgets.Layout(width='280px', height='40px')
)

def download_txt(b):
    if os.path.exists("watchlist.txt"):
        print("⚡ Downloading watchlist.txt...")
        files.download("watchlist.txt")
    else:
        print("watchlist.txt not found.")

def download_excel(b):
    if os.path.exists("weekly_swing_screener_results.xlsx"):
        print("⚡ Downloading weekly_swing_screener_results.xlsx...")
        files.download("weekly_swing_screener_results.xlsx")
    else:
        print("Excel file not found.")

def download_trades(b):
    if os.path.exists("pipeline_trade_log.csv"):
        print("⚡ Downloading pipeline_trade_log.csv...")
        files.download("pipeline_trade_log.csv")
    else:
        print("pipeline_trade_log.csv not found.")

btn_txt.on_click(download_txt)
btn_excel.on_click(download_excel)
btn_trades.on_click(download_trades)

box = widgets.HBox([btn_txt, btn_excel, btn_trades])
display(box)
